# Evolución de redes de alimentación cruzada en Avida

Este notebook reúne todo el flujo de los experimentos evolutivos (apartados 2.4 y 2.5 de Materiales y Métodos):

1. **Universo de interacciones**: red aleatoria con las 72 especies posibles, generada con `get_network_mod.sh`.
2. **Comunidades ancestrales**: submuestreo de 18 especies (25 % del universo), una de ellas elegida al azar para consumir el aporte del quimiostato, comprobando que formen un único componente conexo.
3. **Búsqueda de fundadores**: dinámica ecológica sin mutación con genotipos asignados al azar, repetida hasta que coexistan exactamente 9 especies.
4. **Experimentos evolutivos**: réplicas independientes (semillas 1–10) de 50.000 *updates* a partir de cada comunidad ancestral.
5. **Análisis de fenotipos**: los 2.500 *updates* finales de cada réplica se procesan con `run_evo_mutua_mod.sh`.

**Ficheros externos necesarios**

| Fichero | Carpeta | Uso |
|---|---|---|
| `get_network_mod.sh` | la de este notebook | genera el universo de interacciones |
| `chemostat.py`, `run-avida.sh` y la configuración de Avida | `DIR_CHEMOSTAT` | ejecuciones ecológicas y evolutivas |
| `run_evo_mutua_mod.sh` | `DIR_FENOTIPOS` | análisis de fenotipos |

Todos los resultados se guardan en `DIR_RESULTADOS`, con una carpeta por comunidad ancestral. Como las ejecuciones son largas, con `REANUDAR = True` el notebook salta los pasos cuyos ficheros de salida ya existen, de modo que puede interrumpirse y volver a ejecutarse.

In [ ]:
import re
import shutil
import subprocess
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd

## Parámetros

Este es el único bloque que hay que modificar.

In [ ]:
# ── Rutas ────────────────────────────────────────────────────────────────────
DIR_CHEMOSTAT    = Path("exp_run")                  # chemostat.py, run-avida.sh y la red del universo
DIR_FENOTIPOS    = Path("fenotipos")   # run_evo_mutua_mod.sh
DIR_RESULTADOS   = Path("resultados_evolucion")
SCRIPT_RED       = Path("get_network_mod.sh")
SCRIPT_AVIDA     = "run-avida.sh"             # dentro de DIR_CHEMOSTAT
SCRIPT_FENOTIPOS = "run_evo_mutua_mod.sh"     # dentro de DIR_FENOTIPOS
PYTHON           = "python"                   # intérprete con el que se ejecuta chemostat.py

# ── 1. Universo de interacciones (get_network_mod.sh) ────────────────────────
GENERAR_RED       = False             # False: usa la red existente; True: genera una nueva
RED_UNIVERSO      = "network_50.csv"  # nombre del fichero de la red, dentro de DIR_CHEMOSTAT
CONECTANCIA_MEDIA = 0.5               # conectancia media
CONECTANCIA_SD    = 0.0               # desviación típica de la conectancia
PRODUCCION_MEDIA  = 3.0               # unidades de subproductos liberadas por especie
PRODUCCION_SD     = 0.0               # desviación típica de la producción
SEMILLA_RED       = None              # semilla de la red (obligatoria si GENERAR_RED = True)

# ── 2-3. Comunidades ancestrales ─────────────────────────────────────────────
N_COMUNIDADES            = 10     # comunidades ancestrales que se quieren obtener
TAMANO_SUBMUESTREO       = 18     # especies submuestreadas (25 % de 72)
ESPECIES_OBJETIVO        = 9      # nº exacto de especies que deben coexistir
TIEMPO_ECOLOGICO         = 5000   # updates de la dinámica ecológica sin mutación
INFLOW                   = 30     # unidades de recurso por unidad de tiempo en el quimiostato
MAX_INTENTOS_GENOTIPOS   = 100    # asignaciones de genotipos antes de descartar un submuestreo
MAX_INTENTOS_SUBMUESTREO = 1000   # submuestreos antes de abandonar la búsqueda de componente único
EXIGIR_ENTRADA           = False  # True: toda especie salvo la del quimiostato debe recibir subproductos
SEMILLA_SUBMUESTREO      = 1      # semilla del submuestreo de especies

# ── 4. Experimentos evolutivos ───────────────────────────────────────────────
N_REPLICAS       = 10      # réplicas por comunidad (semillas de Avida 1..N_REPLICAS)
TIEMPO_EVOLUCION = 50000   # updates de cada experimento evolutivo
TASA_MUTACION    = 0.01    # probabilidad de mutación puntual por copia (COPY_MUT_PROB)

# ── 5. Análisis de fenotipos ─────────────────────────────────────────────────
VENTANA_FINAL = 2500       # updates finales que se analizan

# ── Control de la ejecución ──────────────────────────────────────────────────
EJECUTAR = True   # False: solo muestra los comandos, sin ejecutarlos (modo prueba)
REANUDAR = True   # True: salta los pasos cuyos ficheros de salida ya existen

In [ ]:
def ejecutar(comando, carpeta):
    # Muestra el comando y, si EJECUTAR = True, lo ejecuta en la carpeta indicada.
    comando = [str(c) for c in comando]
    print("$", " ".join(comando), f"   [en {carpeta}]")
    if EJECUTAR:
        subprocess.run(comando, cwd=carpeta, check=True)

DIR_RESULTADOS.mkdir(parents=True, exist_ok=True)

def fijar_parametro_avida(parametro, valor):
    # Cambia el valor de un parámetro "-set PARAMETRO valor" en run-avida.sh.
    ruta = DIR_CHEMOSTAT / SCRIPT_AVIDA
    texto, n = re.subn(rf"-set {parametro} [^\s\\]+", f"-set {parametro} {valor}", ruta.read_text())
    if n == 0:
        raise ValueError(f"No se encontró '-set {parametro}' en {ruta}")
    if EJECUTAR:
        ruta.write_text(texto)

## 1. Universo de interacciones

`get_network_mod.sh` genera una red dirigida entre las 72 especies (todas las combinaciones ordenadas de dos funciones booleanas). La conectancia se extrae de una normal con la media y la desviación indicadas; después se reparte al azar la producción de cada especie entre sus consumidores. La red se escribe con tres decimales, de modo que los aportes inferiores a 0,0005 se redondean a cero y desaparecen: por eso la conectancia final puede quedar algo por debajo de la nominal.

In [ ]:
ruta_red = DIR_CHEMOSTAT / RED_UNIVERSO

if GENERAR_RED:
    if SEMILLA_RED is None:
        raise ValueError("Indica SEMILLA_RED para generar la red.")
    ejecutar(["bash", SCRIPT_RED.resolve(), CONECTANCIA_MEDIA, CONECTANCIA_SD,
              PRODUCCION_MEDIA, PRODUCCION_SD, SEMILLA_RED, RED_UNIVERSO], DIR_CHEMOSTAT)

U = pd.read_csv(ruta_red, index_col=0)
A = U.to_numpy()
S = A.shape[0]
enlaces = int((A > 0).sum())
print(f"Especies: {S}")
print(f"Enlaces: {enlaces} de {S * (S - 1)} posibles (conectancia = {enlaces / (S * (S - 1)):.3f})")
print(f"Producción por especie: {A.sum(1).min():.3f} a {A.sum(1).max():.3f}")
print(f"Consumidores por especie: {(A > 0).sum(1).min()} a {(A > 0).sum(1).max()}")
print(f"Productores por especie: {(A > 0).sum(0).min()} a {(A > 0).sum(0).max()}")
print(f"Pares con interacción recíproca: {int(((A > 0) & (A.T > 0)).sum() / 2)}")

## 2. Submuestreo de las comunidades ancestrales

`chemostat.py` identifica las especies con un código de dos letras (AA = 1, …, CT = 72), en el mismo orden que las filas de la red. Cada submuestreo toma 18 especies sin reemplazo, elige una de ellas al azar para consumir el aporte del quimiostato y se acepta solo si las 18 especies forman un único componente conexo en el universo de interacciones, sin tener en cuenta la dirección de los enlaces.

In [ ]:
def num_a_codigo(n):
    # Convierte un número 1-72 en el código de dos letras (1 = AA, 72 = CT).
    return chr(ord("A") + (n - 1) // 26) + chr(ord("A") + (n - 1) % 26)

CODIGO_A_ETIQUETA = {num_a_codigo(i + 1): etiqueta for i, etiqueta in enumerate(U.index)}
G_universo = nx.from_pandas_adjacency(U, create_using=nx.DiGraph)

def submuestrear(rng):
    # Devuelve un submuestreo válido: un único componente conexo y, si se exige, entrada para todas.
    for intento in range(1, MAX_INTENTOS_SUBMUESTREO + 1):
        codigos = sorted(str(c) for c in rng.choice(list(CODIGO_A_ETIQUETA), TAMANO_SUBMUESTREO, replace=False))
        quimiostato = str(rng.choice(codigos))
        etiquetas = [CODIGO_A_ETIQUETA[c] for c in codigos]
        G = G_universo.subgraph(etiquetas)
        if not nx.is_weakly_connected(G):
            continue
        sin_entrada = [n for n in G if G.in_degree(n) == 0 and n != CODIGO_A_ETIQUETA[quimiostato]]
        if EXIGIR_ENTRADA and sin_entrada:
            continue
        return {"codigos": codigos, "quimiostato": quimiostato, "etiquetas": etiquetas,
                "sin_entrada": sin_entrada, "intentos_submuestreo": intento}
    raise RuntimeError("No se encontró ningún submuestreo válido; revisa los parámetros.")

# Ejemplo: un submuestreo con la semilla fijada
ejemplo = submuestrear(np.random.default_rng([SEMILLA_SUBMUESTREO, 0]))
print("Códigos:", ",".join(ejemplo["codigos"]))
print("Especie del quimiostato:", ejemplo["quimiostato"], "=", CODIGO_A_ETIQUETA[ejemplo["quimiostato"]])
print("Especies sin productores en el submuestreo:", ejemplo["sin_entrada"] or "ninguna")

## 3. Búsqueda de fundadores

Para cada submuestreo, `chemostat.py` asigna a cada especie un genotipo al azar de la base de datos y simula la dinámica ecológica sin mutación durante `TIEMPO_ECOLOGICO` *updates*. Se cuentan las especies presentes al final y, si son exactamente `ESPECIES_OBJETIVO`, se guarda el fichero de fundadores (`founders.csv`) con los genotipos usados. Si tras `MAX_INTENTOS_GENOTIPOS` asignaciones no se alcanza el objetivo, el submuestreo se descarta y se toma otro.

Cada comunidad ancestral usa su propio generador aleatorio (derivado de `SEMILLA_SUBMUESTREO` y del número de comunidad), así que al reanudar se obtienen los mismos submuestreos.

In [ ]:
def buscar_fundadores(sub, destino):
    # Ejecuta la dinámica ecológica hasta que coexistan ESPECIES_OBJETIVO especies.
    comando = [PYTHON, "chemostat.py", f"--adj_matrix={RED_UNIVERSO}", "--phenotype_matrix",
               f"--initial_organisms={','.join(sub['codigos'])}", f"--exp_time={TIEMPO_ECOLOGICO}",
               f"--inflow={INFLOW}", f"--chemo_resources={sub['quimiostato']}"]
    fijar_parametro_avida("COPY_MUT_PROB", 0)   # fase ecológica: sin mutación
    for intento in range(1, MAX_INTENTOS_GENOTIPOS + 1):
        ejecutar(comando, DIR_CHEMOSTAT)
        if not EJECUTAR:
            return None
        abundancias = pd.read_csv(DIR_CHEMOSTAT / "abundances.csv")
        n = int((abundancias["update"] == TIEMPO_ECOLOGICO).sum())
        print(f"  Asignación {intento}: {n} especies coexisten en el update {TIEMPO_ECOLOGICO}")
        if n == ESPECIES_OBJETIVO:
            shutil.copy(DIR_CHEMOSTAT / "founders.csv", destino)
            return intento
    return None

ruta_registro = DIR_RESULTADOS / "comunidades_ancestrales.csv"
registro = pd.read_csv(ruta_registro).to_dict("records") if (REANUDAR and ruta_registro.exists()) else []
usados = {r["codigos"] for r in registro}

while len(registro) < N_COMUNIDADES:
    k = len(registro) + 1
    rng = np.random.default_rng([SEMILLA_SUBMUESTREO, k])
    carpeta = DIR_RESULTADOS / f"comunidad_{k:02d}"
    carpeta.mkdir(exist_ok=True)
    fundadores = carpeta / f"founders_comunidad_{k:02d}.csv"
    print(f"\n=== Comunidad ancestral {k} ===")
    while True:
        sub = submuestrear(rng)
        if ",".join(sub["codigos"]) in usados:
            continue
        print("Submuestreo:", ",".join(sub["codigos"]), "| quimiostato:", sub["quimiostato"])
        intentos = buscar_fundadores(sub, fundadores)
        if not EJECUTAR or intentos is not None:
            break
        print("  Submuestreo descartado: no se alcanzó el número de especies objetivo.")
    if not EJECUTAR:
        print("\nModo prueba: la búsqueda se detiene aquí.")
        break
    registro.append({"comunidad": k, "codigos": ",".join(sub["codigos"]),
                     "quimiostato": sub["quimiostato"],
                     "especie_quimiostato": CODIGO_A_ETIQUETA[sub["quimiostato"]],
                     "etiquetas": ",".join(sub["etiquetas"]),
                     "intentos_submuestreo": sub["intentos_submuestreo"],
                     "intentos_genotipos": intentos, "fundadores": str(fundadores)})
    usados.add(",".join(sub["codigos"]))
    pd.DataFrame(registro).to_csv(ruta_registro, index=False)
    print(f"  Fundadores guardados en {fundadores}")

pd.DataFrame(registro)

## 4. Experimentos evolutivos

Para cada comunidad ancestral se ejecutan `N_REPLICAS` réplicas con semillas de Avida 1, 2, …, cambiando `RANDOM_SEED` en `run-avida.sh` antes de cada una. Cada réplica parte del fichero de fundadores y dura `TIEMPO_EVOLUCION` *updates*. La salida `abundances.csv` se guarda como `abundances_seed_<semilla>.csv` en la carpeta de la comunidad.

`run-avida.sh` conserva la última semilla usada, que también afectará a las búsquedas de fundadores posteriores.

In [ ]:
def fijar_semilla_avida(semilla):
    # Cambia el valor de RANDOM_SEED en run-avida.sh.
    fijar_parametro_avida("COPY_MUT_PROB", TASA_MUTACION)   # fase evolutiva
    ruta = DIR_CHEMOSTAT / SCRIPT_AVIDA
    texto, n = re.subn(r"-set RANDOM_SEED \d+", f"-set RANDOM_SEED {semilla}", ruta.read_text())
    if n == 0:
        raise ValueError(f"No se encontró '-set RANDOM_SEED' en {ruta}")
    if EJECUTAR:
        ruta.write_text(texto)

for c in registro:
    k = int(c["comunidad"])
    carpeta = DIR_RESULTADOS / f"comunidad_{k:02d}"
    shutil.copy(c["fundadores"], DIR_CHEMOSTAT / Path(c["fundadores"]).name)
    for semilla in range(1, N_REPLICAS + 1):
        salida = carpeta / f"abundances_seed_{semilla}.csv"
        if REANUDAR and salida.exists():
            continue
        print(f"\n=== Comunidad {k}, réplica (semilla) {semilla} ===")
        fijar_semilla_avida(semilla)
        ejecutar([PYTHON, "chemostat.py", f"--adj_matrix={RED_UNIVERSO}", "--phenotype_matrix",
                  f"--initial_organisms={c['codigos']}", f"--chemo_resources={c['quimiostato']}",
                  f"--exp_time={TIEMPO_EVOLUCION}", f"--inflow={INFLOW}",
                  f"--organisms_path={Path(c['fundadores']).name}"], DIR_CHEMOSTAT)
        if EJECUTAR:
            shutil.copy(DIR_CHEMOSTAT / "abundances.csv", salida)
            print(f"  Guardado en {salida}")

## 5. Análisis de fenotipos

De cada réplica se extraen las filas de los últimos `VENTANA_FINAL` *updates* (de `TIEMPO_EVOLUCION − VENTANA_FINAL` a `TIEMPO_EVOLUCION`, ambos incluidos), se procesan con `run_evo_mutua_mod.sh` y el resultado se guarda como `population_phenotype_rep<comunidad>_def_seed_<semilla>.csv`. El filtrado se hace sobre el texto del fichero, sin reescribir los números, para que `run_evo_mutua_mod.sh` reciba las filas exactamente como las escribió `chemostat.py`.

In [ ]:
def extraer_ventana(origen, destino, inicio, fin):
    # Copia la cabecera y las filas cuyo primer campo (update) está entre inicio y fin.
    with open(origen) as f_in, open(destino, "w") as f_out:
        f_out.write(f_in.readline())
        for linea in f_in:
            update = float(linea.split(",", 1)[0])
            if inicio <= update <= fin:
                f_out.write(linea)

for c in registro:
    k = int(c["comunidad"])
    carpeta = DIR_RESULTADOS / f"comunidad_{k:02d}"
    for semilla in range(1, N_REPLICAS + 1):
        salida = carpeta / f"population_phenotype_rep{k}_def_seed_{semilla}.csv"
        if REANUDAR and salida.exists():
            continue
        abundancias = carpeta / f"abundances_seed_{semilla}.csv"
        if not abundancias.exists():
            print(f"Falta {abundancias}; se omite.")
            continue
        print(f"\n=== Fenotipos: comunidad {k}, semilla {semilla} ===")
        extraer_ventana(abundancias, DIR_FENOTIPOS / "selected_abundances.csv",
                        TIEMPO_EVOLUCION - VENTANA_FINAL, TIEMPO_EVOLUCION)
        ejecutar(["bash", SCRIPT_FENOTIPOS, "selected_abundances.csv"], DIR_FENOTIPOS)
        if EJECUTAR:
            shutil.move(DIR_FENOTIPOS / "population_phenotype.csv", salida)
            print(f"  Guardado en {salida}")

## Resumen de los ficheros generados

In [ ]:
resumen = []
for c in registro:
    k = int(c["comunidad"])
    carpeta = DIR_RESULTADOS / f"comunidad_{k:02d}"
    resumen.append({"comunidad": k, "especie_quimiostato": c["especie_quimiostato"],
                    "replicas_evolucion": len(list(carpeta.glob("abundances_seed_*.csv"))),
                    "replicas_fenotipos": len(list(carpeta.glob("population_phenotype_*.csv")))})
pd.DataFrame(resumen)